# AOOP — Colab Ollama para OpenClaw / NPC Society

Este notebook serve para correr o **Ollama no Google Colab** e expor a API para o projeto Unity local através de um **Cloudflare Tunnel**.

Fluxo usado no trabalho:

```text
Unity local
→ Flask Bridge local :5055
→ OpenClaw Gateway local :18789
→ proxy local :11434
→ Cloudflare Tunnel
→ Ollama no Colab
→ qwen3:4b-instruct
```

Versão estável do projeto:

```text
branch: main
commit: cef4c0e
stable tag: stable-society-scheduler-selection-context
```

Este notebook foi reorganizado para conter apenas os passos necessários. Foram removidos comandos temporários, testes com URLs antigos e células de tentativa/erro.


## Ordem de execução

No Colab:

1. Verificar GPU.
2. Instalar dependências e Ollama.
3. Arrancar Ollama.
4. Instalar o modelo `qwen3:4b-instruct`.
5. Aquecer/testar o modelo.
6. Instalar `cloudflared`.
7. Criar o túnel Cloudflare com `--http-host-header localhost:11434`.
8. Copiar o `CLOUDFLARE_URL`.

No Windows:

1. Parar serviços antigos nas portas `11434`, `18789` e `5055`.
2. Apontar o proxy local para o URL do Cloudflare.
3. Arrancar o proxy local.
4. Arrancar o OpenClaw Gateway.
5. Arrancar o Flask Bridge.
6. Abrir Unity e ativar a autonomia.

Regra importante: se a sessão do Colab cair, tens de repetir a parte do Colab e copiar o novo `CLOUDFLARE_URL` para o Windows.


# Parte 1 — Google Colab

As células desta parte são para executar no Colab.


## 1. Verificar GPU

O ideal é aparecer uma GPU **Tesla T4**. Se estiver em CPU, muda em:

```text
Runtime / Tempo de execução → Change runtime type → T4 GPU → Save
```


In [1]:
!nvidia-smi


Tue Jun 16 21:00:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Instalar dependências e Ollama

Esta célula instala o Ollama e ferramentas necessárias no runtime atual do Colab. Sempre que o runtime reiniciar, esta célula tem de ser executada novamente.


In [2]:
%%bash
set -e

apt-get update -qq
apt-get install -y zstd curl wget

curl -fsSL https://ollama.com/install.sh | sh

echo ""
echo "=== Ollama binary ==="
which ollama

echo ""
echo "=== Ollama version ==="
ollama --version || true


Reading package lists...
Building dependency tree...
Reading state information...
curl is already the newest version (7.81.0-1ubuntu1.24).
wget is already the newest version (1.21.2-2ubuntu1.1).
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 86 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (9,892 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...

=== Ollama binary ===
/usr/local/bin/ollama

=== Ollama version ===


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


Notas esperadas:

- O aviso `systemd is not running` é normal no Colab.
- O aviso `could not connect to a running Ollama instance` pode aparecer antes de arrancar o servidor. Não é erro crítico nesta fase.


## 3. Arrancar Ollama

Esta célula arranca o serviço Ollama em background na porta `11434`.

Configuração usada:

```text
OLLAMA_HOST=0.0.0.0:11434
OLLAMA_NUM_PARALLEL=1
OLLAMA_MAX_LOADED_MODELS=1
OLLAMA_KEEP_ALIVE=30m
OLLAMA_CONTEXT_LENGTH=32768
```

O contexto alto ajuda nos pedidos do OpenClaw, mas a lógica do backend também reduz o contexto enviado ao modelo.


In [3]:
%%bash
set +e

pkill -f "ollama serve" || true
pkill -f "llama-server" || true
sleep 2

rm -f /content/ollama_colab.log
rm -f /content/ollama.pid
rm -f /tmp/ollama_tags.json

export OLLAMA_HOST=0.0.0.0:11434
export OLLAMA_KV_CACHE_TYPE=q8_0
export OLLAMA_NUM_PARALLEL=1
export OLLAMA_MAX_LOADED_MODELS=1
export OLLAMA_KEEP_ALIVE=30m
export OLLAMA_CONTEXT_LENGTH=32768
export OLLAMA_DEBUG=1

nohup ollama serve > /content/ollama_colab.log 2>&1 &
echo $! > /content/ollama.pid

echo "Waiting for Ollama on http://127.0.0.1:11434 ..."
for i in $(seq 1 60); do
    if curl -s http://127.0.0.1:11434/api/tags > /tmp/ollama_tags.json; then
        echo "Ollama is ready."
        cat /tmp/ollama_tags.json
        exit 0
    fi
    sleep 1
done

echo "Ollama did not become ready. Log tail:"
tail -n 120 /content/ollama_colab.log || true
exit 1


Waiting for Ollama on http://127.0.0.1:11434 ...
Ollama is ready.
{"models":[]}

Resultado esperado antes de instalar o modelo:

```json
{"models":[]}
```


## 4. Instalar o modelo validado

Modelo usado no trabalho:

```text
qwen3:4b-instruct
```


In [4]:
%%bash
set -e

ollama pull qwen3:4b-instruct


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest 
pulling 85e4a5b7b8ef:   0% ▕                  ▏ 2.3 MB/2.5 GB                  pulling manifest 
pulling 85e4a5b7b8ef:   3% ▕                  ▏  82 MB/2.5 GB                  pulling manifest 
pulling 85e4a5b7b8ef:   6% ▕█                 ▏ 139 MB/2.5 GB                  pulling manifest 
pulling 85e4a5b7b8ef:   6% ▕█                 ▏ 158 MB/2.5 GB                  pulling manifest 
pulling 85e4a5b7b8ef:   8% ▕█                 ▏ 205 MB/2.5 GB                  pulling manifest 
pulling 85e4a5b7b8ef:  10% ▕█                 ▏ 253 MB/2.5 GB                  pulling manifest 
pulling 85e4a5b7b8ef:  11% ▕██                ▏ 283 MB/2.5 GB                  pulling manifest 
pulling 85e4a5b7b8ef:  14% ▕██                ▏ 344 MB/2.5 GB                  pulling manifest 
pulling 85e4a5b7

## 5. Aquecer o modelo e confirmar que está carregado

O primeiro pedido pode demorar. É normal.


In [5]:
%%bash
set -e

curl -s http://127.0.0.1:11434/api/chat \
  -H "Content-Type: application/json" \
  -d '{"model":"qwen3:4b-instruct","messages":[{"role":"user","content":"Respond only with OK."}],"stream":false}' \
  | python3 -m json.tool

echo ""
echo "=== Loaded models ==="
ollama ps


Process is interrupted.


Confirma que aparece algo equivalente a:

```json
"content": "OK"
```

E que `ollama ps` mostra o modelo carregado.


## 6. Testar a API local do Ollama

Este teste confirma que o endpoint usado pelo proxy local responde corretamente dentro do Colab.


In [10]:
%%bash
cat > /tmp/test_ollama.json <<'JSON'
{
  "model": "qwen3:4b-instruct",
  "messages": [
    {
      "role": "user",
      "content": "Respond only with OK."
    }
  ],
  "stream": false
}
JSON

curl -i -X POST http://127.0.0.1:11434/api/chat \
  -H "Content-Type: application/json" \
  -d @/tmp/test_ollama.json

HTTP/1.1 200 OK
Content-Type: application/json; charset=utf-8
Date: Tue, 16 Jun 2026 21:07:16 GMT
Content-Length: 308

{"model":"qwen3:4b-instruct","created_at":"2026-06-16T21:07:16.559857486Z","message":{"role":"assistant","content":"OK"},"done":true,"done_reason":"stop","total_duration":33939608143,"load_duration":3516912992,"prompt_eval_count":13,"prompt_eval_duration":30390433000,"eval_count":2,"eval_duration":22837000}

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   448  100   308  100   140      9      4  0:00:35  0:00:33  0:00:02    65


## 7. Ver logs rápidos do Ollama

Usar esta célula só para diagnóstico.


In [ ]:
%%bash
set +e

echo "=== Ollama process ==="
ps aux | grep -E "ollama|llama-server" | grep -v grep || true

echo ""
echo "=== Ollama loaded models ==="
ollama ps || true

echo ""
echo "=== Ollama log tail ==="
tail -n 120 /content/ollama_colab.log || true


## 8. Instalar Cloudflared

Esta célula instala o binário do Cloudflare Tunnel no runtime atual.


In [11]:
%%bash
set -e

rm -f /content/cloudflared
wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
chmod +x /content/cloudflared
/content/cloudflared --version


cloudflared version 2026.6.0 (built 2026-06-09-09:33 UTC)


## 9. Criar túnel Cloudflare robusto

Esta é a célula principal para expor o Ollama do Colab.

Ponto importante: esta versão usa:

```text
--http-host-header localhost:11434
```

Isto evita o erro `HTTP 403 Forbidden` que apareceu quando o Cloudflare não encaminhava o `Host` corretamente.

A célula termina depois de encontrar o URL. O processo `cloudflared` fica em background enquanto o runtime do Colab estiver ativo.


In [12]:
import subprocess
import time
import re
from pathlib import Path

log_path = Path("/content/cloudflared.log")
url_path = Path("/content/cloudflare_url.txt")

subprocess.run("pkill -f cloudflared || true", shell=True)
time.sleep(3)

for path in (log_path, url_path):
    if path.exists():
        path.unlink()

cmd = (
    "/content/cloudflared tunnel "
    "--url http://127.0.0.1:11434 "
    "--http-host-header localhost:11434 "
    "--no-autoupdate "
    "--loglevel info "
    "> /content/cloudflared.log 2>&1 &"
)

subprocess.run(cmd, shell=True)

url = None
for _ in range(60):
    time.sleep(1)
    text = log_path.read_text(errors="ignore") if log_path.exists() else ""
    match = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", text)
    if match:
        url = match.group(0)
        break

print("CLOUDFLARE_URL=", url)

if url:
    url_path.write_text(url, encoding="utf-8")
else:
    print("No Cloudflare URL was found.")

print("--- cloudflared log tail ---")
print(log_path.read_text(errors="ignore")[-3000:] if log_path.exists() else "No log file.")


CLOUDFLARE_URL= https://integrating-carey-penn-authentic.trycloudflare.com
--- cloudflared log tail ---
2026-06-16T21:07:30Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-06-16T21:07:30Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-06-16T21:07:32Z INF +--------------------------------------------------------------------------------------------+
2026-06-16T21:07:32Z INF |  Your quick Tunnel has been created! Visit it at (it may take some tim

## 10. Testar o túnel Cloudflare

Se esta célula devolver `HTTP/2 200` e `content: OK`, o Colab está pronto para ser usado pelo Windows.

Se devolver `403 Forbidden`, recria o túnel executando outra vez a célula anterior. Se continuar a falhar, reinicia o runtime e repete a parte do Colab.


In [13]:
%%bash
set -e

CF=$(cat /content/cloudflare_url.txt)
echo "CLOUDFLARE_URL=$CF"

echo ""
echo "=== Tunnel /api/tags ==="
curl -i "$CF/api/tags"

echo ""
echo "=== Tunnel /api/chat ==="
curl -i -X POST "$CF/api/chat" \
  -H "Content-Type: application/json" \
  -d '{"model":"qwen3:4b-instruct","messages":[{"role":"user","content":"Respond only with OK."}],"stream":false}'


CLOUDFLARE_URL=https://integrating-carey-penn-authentic.trycloudflare.com

=== Tunnel /api/tags ===
HTTP/2 200 
date: Tue, 16 Jun 2026 21:07:44 GMT
content-type: application/json; charset=utf-8
content-length: 434
cf-ray: a0ccc92b994c0907-SEA
cf-cache-status: DYNAMIC
server: cloudflare

{"models":[{"name":"qwen3:4b-instruct","model":"qwen3:4b-instruct","modified_at":"2026-06-16T21:03:12.605002138Z","size":2497293803,"digest":"0edcdef34593eac1aa2be9c7d06c432dcf81945adca5eca2f27662c18f168ba0","details":{"parent_model":"","format":"gguf","family":"qwen3","families":["qwen3"],"parameter_size":"4.0B","quantization_level":"Q4_K_M","context_length":262144,"embedding_length":2560},"capabilities":["completion","tools"]}]}
=== Tunnel /api/chat ===
HTTP/2 200 
date: Tue, 16 Jun 2026 21:07:45 GMT
content-type: application/json; charset=utf-8
content-length: 302
cf-ray: a0ccc92ea9b2d301-SEA
cf-cache-status: DYNAMIC
server: cloudflare

{"model":"qwen3:4b-instruct","created_at":"2026-06-16T21:07:45.4

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   434  100   434    0     0    868      0 --:--:-- --:--:-- --:--:--   868
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   409  100   302  100   107    662    234 --:--:-- --:--:-- --:--:--   898


## 11. Monitorizar ou parar o túnel

Estas células são opcionais. Usa apenas para diagnóstico ou para trocar de túnel.


In [ ]:
%%bash
set +e

echo "=== Cloudflared process ==="
ps aux | grep cloudflared | grep -v grep || true

echo ""
echo "=== Current Cloudflare URL ==="
cat /content/cloudflare_url.txt 2>/dev/null || true

echo ""
echo "=== Cloudflared log tail ==="
tail -n 120 /content/cloudflared.log || true


In [ ]:
%%bash
set +e

pkill -f cloudflared || true
sleep 2
ps aux | grep cloudflared | grep -v grep || true


# Parte 2 — Windows / PowerShell

Os comandos seguintes **não são para executar no Colab**. São para copiar para o PowerShell no Windows.

Assume-se que o projeto está em:

```powershell
C:\Uni\3º ano\2ºSemestre\AO\TP4\2026-ei-aoopii-a18-clean
```


## A. Confirmar versão estável do projeto

```powershell
cd "C:\Uni\3º ano\2ºSemestre\AO\TP4\2026-ei-aoopii-a18-clean"

git fetch --all --tags
git status -sb
git log -5 --oneline --decorate
git tag --list "stable-*"
```

Esperado no topo do log:

```text
cef4c0e (HEAD -> main, tag: stable-society-scheduler-selection-context, origin/main, origin/HEAD) Improve OpenClaw society scheduler selection context
```


## B. Limpar serviços antigos nas portas locais

Executa antes de uma demo, especialmente se o jogo, proxy ou Gateway ficaram abertos de uma sessão anterior.

```powershell
cd "C:\Uni\3º ano\2ºSemestre\AO\TP4\2026-ei-aoopii-a18-clean"

Write-Host "Stopping local proxy/Ollama on 11434 if present..."
$pid11434 = Get-NetTCPConnection `
    -LocalPort 11434 `
    -State Listen `
    -ErrorAction SilentlyContinue |
    Select-Object -ExpandProperty OwningProcess -First 1

if ($pid11434) {
    Stop-Process -Id $pid11434 -Force
}

Write-Host "Stopping OpenClaw Gateway if present..."
openclaw gateway stop

$pid18789 = Get-NetTCPConnection `
    -LocalPort 18789 `
    -State Listen `
    -ErrorAction SilentlyContinue |
    Select-Object -ExpandProperty OwningProcess -First 1

if ($pid18789) {
    Stop-Process -Id $pid18789 -Force
}

Write-Host "Stopping Flask Bridge on 5055 if present..."
$pid5055 = Get-NetTCPConnection `
    -LocalPort 5055 `
    -State Listen `
    -ErrorAction SilentlyContinue |
    Select-Object -ExpandProperty OwningProcess -First 1

if ($pid5055) {
    Stop-Process -Id $pid5055 -Force
}

Start-Sleep -Seconds 2

Write-Host "=== Remaining ports ==="
Get-NetTCPConnection -LocalPort 11434,18789,5055 -State Listen -ErrorAction SilentlyContinue
```

Se não aparecer nada no fim, as portas ficaram livres.


## C. Arrancar o proxy local para o Colab

Num PowerShell novo:

```powershell
cd "C:\Uni\3º ano\2ºSemestre\AO\TP4\2026-ei-aoopii-a18-clean"

$env:COLAB_OLLAMA_URL = "https://COLOCAR_AQUI_O_URL_DO_COLAB.trycloudflare.com"

python .\colab_ollama_proxy.py
```

Exemplo:

```powershell
$env:COLAB_OLLAMA_URL = "https://agenda-organized-congress-recreational.trycloudflare.com"
python .\colab_ollama_proxy.py
```

Resultado esperado:

```text
Proxy listening on http://127.0.0.1:11434
Forwarding to https://...
```

Este terminal fica aberto.


## D. Testar o proxy local

Noutro PowerShell:

```powershell
Invoke-RestMethod "http://127.0.0.1:11434/api/tags" |
    ConvertTo-Json -Depth 6
```

Tem de aparecer:

```text
qwen3:4b-instruct
```

Depois testa o chat:

```powershell
$body = @{
    model = "qwen3:4b-instruct"
    messages = @(
        @{
            role = "user"
            content = "Respond only with OK."
        }
    )
    stream = $false
} | ConvertTo-Json -Depth 8

Invoke-RestMethod `
    -Uri "http://127.0.0.1:11434/api/chat" `
    -Method Post `
    -ContentType "application/json" `
    -Body $body |
    ConvertTo-Json -Depth 10
```

Tem de aparecer:

```text
"content": "OK"
```

Se isto falhar, ainda não avances para OpenClaw nem Unity.


## E. Registar o society agent no OpenClaw

Executar uma vez depois de mudar/recriar configuração OpenClaw:

```powershell
cd "C:\Uni\3º ano\2ºSemestre\AO\TP4\2026-ei-aoopii-a18-clean"

& ".\scripts\register_openclaw_society_agent.ps1"
```

Resultado esperado:

```text
Society agent registered successfully.
Agent route: openclaw/society
Model: ollama-npc-qwen3/qwen3:4b-instruct
```


## F. Arrancar OpenClaw Gateway

Num PowerShell novo:

```powershell
cd "C:\Uni\3º ano\2ºSemestre\AO\TP4\2026-ei-aoopii-a18-clean"

openclaw gateway run --force
```

Resultado esperado:

```text
[gateway] ready
[heartbeat] started
```

Este terminal fica aberto.


## G. Arrancar Flask Bridge

Num PowerShell novo:

```powershell
cd "C:\Uni\3º ano\2ºSemestre\AO\TP4\2026-ei-aoopii-a18-clean\OpenClaw_Server"

$env:OPENCLAW_TIMEOUT_SECONDS = "420"
$env:OPENCLAW_TEMPERATURE = "0"
$env:OPENCLAW_MAX_OUTPUT_TOKENS = "300"
$env:OPENCLAW_REPAIR_ENABLED = "true"
$env:OPENCLAW_FALLBACK_ENABLED = "true"
$env:OPENCLAW_STRICT_SCHEMA = "true"

python app.py
```

Resultado esperado:

```text
Running on http://127.0.0.1:5055
```

Este terminal fica aberto.


## H. Validar Flask Bridge e scheduler

Noutro PowerShell, na raiz do projeto:

```powershell
cd "C:\Uni\3º ano\2ºSemestre\AO\TP4\2026-ei-aoopii-a18-clean"

Invoke-RestMethod http://127.0.0.1:5055/health |
    ConvertTo-Json -Depth 8
```

Esperado:

```text
"status": "ok"
"gatewayAvailable": true
"configuredSocietySchedulerRoute": "openclaw/society"
```

Teste manual do scheduler:

```powershell
Invoke-RestMethod `
    -Uri "http://127.0.0.1:5055/choose_next_actor" `
    -Method Post `
    -ContentType "application/json" `
    -InFile ".\docs\scheduler_test_payload.json" |
    ConvertTo-Json -Depth 10
```

Esperado:

```text
"decisionSource": "openclaw_society_scheduler"
"actorName": "Lia" ou outro NPC válido
"priority": número entre 0 e 10
```


# Parte 3 — Unity

Abrir a cena:

```text
src/NPCSociety3D/Assets/Scenes/MainScene.unity
```

Selecionar o objeto:

```text
Managers
```

No componente `Open Claw Society Autonomy Manager`, confirmar:

```text
Actor Selection Mode: Open Claw Society Scheduler
Society Scheduler URL: http://127.0.0.1:5055/choose_next_actor
Society Scheduler Timeout Seconds: 60
Fallback To Round Robin On Scheduler Failure: ligado
Agent Controllers: 4
Start Agent Index: 0
Max Autonomous Agents: 4
Max Decisions Per Play Session: 4
Disable Autonomy After Decision Limit: ligado
Initial Delay Seconds: 10
Global Cooldown Seconds: 20
Npc Cooldown Seconds: 30
Max Decisions Per Minute: 1
Enable Debug Logs: ligado
```

Fluxo de teste:

```text
1. Garantir que Enable Autonomy está desligado.
2. Carregar Play.
3. Esperar aparecer: OpenClaw autonomy manager found 4 agent controller(s).
4. Ligar Enable Autonomy.
5. Esperar pela decisão.
```

Logs esperados:

```text
Society scheduler selected actor ... | Source: openclaw_society_scheduler
Autonomous OpenClaw decision requested for ...
OpenClaw request started for ...
OpenClaw decision received | Source: openclaw_agent | Route: openclaw/...
OpenClaw decision executed successfully
Autonomous OpenClaw decision finished for ...
```

Quando atingir o limite:

```text
OpenClaw society autonomy disabled after reaching the play session decision limit.
```


# Parte 4 — Como parar tudo

## Unity

Sair do Play Mode.

## Flask Bridge

No terminal onde está `python app.py`:

```text
CTRL+C
```

## OpenClaw Gateway

```powershell
openclaw gateway stop
```

Se não fechar:

```powershell
Get-NetTCPConnection -LocalPort 18789 -ErrorAction SilentlyContinue |
    Select-Object -ExpandProperty OwningProcess |
    ForEach-Object { Stop-Process -Id $_ -Force }
```

## Proxy local

No terminal do proxy:

```text
CTRL+C
```

Se necessário:

```powershell
Get-NetTCPConnection -LocalPort 11434 -ErrorAction SilentlyContinue |
    Select-Object -ExpandProperty OwningProcess |
    ForEach-Object { Stop-Process -Id $_ -Force }
```

## Flask preso na porta 5055

```powershell
Get-NetTCPConnection -LocalPort 5055 -ErrorAction SilentlyContinue |
    Select-Object -ExpandProperty OwningProcess |
    ForEach-Object { Stop-Process -Id $_ -Force }
```


# Parte 5 — Diagnóstico rápido

## Cloudflare devolve 403 Forbidden

Causa provável: túnel sem `--http-host-header localhost:11434`, URL antigo, ou tunnel temporário bloqueado.

Solução:

```text
1. No Colab, parar cloudflared.
2. Executar outra vez a célula "Criar túnel Cloudflare robusto".
3. Testar com a célula "Testar o túnel Cloudflare".
4. Copiar o novo URL para o PowerShell local.
```

## Proxy local devolve 502

Causas prováveis:

```text
- COLAB_OLLAMA_URL errado.
- Célula do túnel caiu.
- Runtime do Colab reiniciou.
- Ollama não está a responder no Colab.
```

Validação:

```powershell
Invoke-RestMethod "http://127.0.0.1:11434/api/tags"
```

## OpenClaw diz que não encontra Ollama

O OpenClaw espera encontrar o Ollama em:

```text
http://127.0.0.1:11434
```

No nosso caso, essa porta é o proxy local. Portanto, primeiro valida o proxy.

## Unity faz fallback em vez de scheduler

Verificar no terminal do Flask Bridge. Se aparecer erro do scheduler, o fallback protege o jogo, mas o scheduler social não respondeu corretamente.

## Colab fica sem GPU

É limite normal do Colab. Pode ser preciso esperar algumas horas. Não há tempo fixo garantido. Alternativas: usar CPU só para testes pequenos, outra conta, Colab Pro/Pay As You Go, ou máquina local com GPU.


# Parte 6 — Checklist de demo

Antes de mostrar:

```text
1. Colab com GPU ativo.
2. Ollama responde localmente no Colab.
3. Cloudflare Tunnel responde com HTTP 200.
4. Proxy local responde em 127.0.0.1:11434.
5. OpenClaw Gateway está ready.
6. Flask Bridge /health devolve status ok.
7. Unity tem Actor Selection Mode = Open Claw Society Scheduler.
8. Enable Autonomy começa desligado.
9. Play na Unity.
10. Ativar Enable Autonomy.
11. Confirmar logs de scheduler + agente individual + execução.
```

Versão final segura:

```text
commit: cef4c0e
tag: stable-society-scheduler-selection-context
```
